In [0]:
USE CATALOG tibia_analytics;
USE SCHEMA gold;
SET TIME ZONE 'UTC';

In [0]:
CREATE OR REPLACE TABLE tibia_analytics.gold.tmp_characters_behavior_daily AS
WITH window_data AS (
  SELECT *
    FROM tibia_analytics.silver.characters_history_enriched
   WHERE source_file_date >= (
           SELECT DATE_SUB(COALESCE(MAX(snapshot_date), DATE '1997-01-07'), 90)
             FROM tibia_analytics.gold.characters_behavior_daily
         )
),  
base AS (
  SELECT current.source_file_date AS snapshot_date,
         current.character_id,
         current.world_id,
         current.account_status,
         current.sex,
         current.vocation_normalized AS vocation,
         current.level,
         current.level - past_1d.level  AS level_delta_1d,
         current.level - past_7d.level  AS level_delta_7d,
         current.level - past_30d.level AS level_delta_30d,
         current.level - past_90d.level AS level_delta_90d,
         MIN(current.source_file_date) OVER(PARTITION BY current.character_id)   AS first_seen_date,
         CAST(current.last_login_at AS DATE) AS last_login_date,
         DATEDIFF(current.source_file_date, CAST(current.last_login_at AS DATE)) AS days_since_last_login,
         current.last_login_at >= current.source_file_date - INTERVAL 1 DAY      AS is_active_1d,
         current.last_login_at >= current.source_file_date - INTERVAL 7 DAY      AS is_active_7d,
         current.last_login_at >= current.source_file_date - INTERVAL 30 DAY     AS is_active_30d,
         current.last_login_at >= current.source_file_date - INTERVAL 90 DAY     AS is_active_90d,
         current.guild IS NOT NULL      AS in_guild,
         current.houses IS NOT NULL     AS owns_house,
         current.married_to IS NOT NULL AS is_married,
         current.account_information.loyalty_title,
         current.achievement_points
    FROM window_data AS current
    LEFT JOIN window_data AS past_1d
      ON past_1d.character_id = current.character_id 
     AND past_1d.source_file_date = DATE_SUB(current.source_file_date, 1)
    LEFT JOIN window_data AS past_7d
      ON past_7d.character_id = current.character_id 
     AND past_7d.source_file_date = DATE_SUB(current.source_file_date, 7) 
    LEFT JOIN window_data AS past_30d 
      ON past_30d.character_id = current.character_id 
     AND past_30d.source_file_date = DATE_SUB(current.source_file_date, 30) 
    LEFT JOIN window_data AS past_90d 
      ON past_90d.character_id = current.character_id 
     AND past_90d.source_file_date = DATE_SUB(current.source_file_date, 90)
),
status_calc AS (
  SELECT snapshot_date,
         character_id,
         world_id,
         account_status,
         sex,
         vocation,
         level,
         level_delta_1d,
         level_delta_7d,
         level_delta_30d,
         level_delta_90d,
         first_seen_date,
         last_login_date,
         days_since_last_login,
         is_active_1d,
         is_active_7d,
         is_active_30d,
         is_active_90d,
         CASE
           WHEN days_since_last_login <= 7 THEN 'active'
           WHEN days_since_last_login <= 30 THEN 'dormant'
           ELSE 'churned'
         END AS base_status,
         in_guild,
         owns_house,
         is_married,
         loyalty_title,
         achievement_points
    FROM base
),
stage_calc AS (
  SELECT snapshot_date,
         character_id,
         world_id,
         account_status,
         sex,
         vocation,
         level,
         level_delta_1d,
         level_delta_7d,
         level_delta_30d,
         level_delta_90d,
         first_seen_date,
         last_login_date,
         days_since_last_login,
         is_active_1d,
         is_active_7d,
         is_active_30d,
         is_active_90d,
         CASE
           WHEN DATEDIFF(snapshot_date, first_seen_date) = 0 THEN 'new'
           WHEN base_status = 'active' AND LAG(base_status) OVER(PARTITION BY character_id ORDER BY snapshot_date) IN ('dormant', 'churned') THEN 'returning'
           WHEN base_status = 'active' THEN 'active'
           WHEN base_status = 'dormant' THEN 'dormant'
           ELSE 'churned'
         END AS lifecycle_stage,
         in_guild,
         owns_house,
         is_married,
         loyalty_title,
         achievement_points
     FROM status_calc
)
SELECT snapshot_date,
       character_id,
       world_id,
       account_status,
       sex,
       vocation,
       level,
       level_delta_1d,
       level_delta_7d,
       level_delta_30d,
       level_delta_90d,
       first_seen_date,
       last_login_date,
       days_since_last_login,
       is_active_1d,
       is_active_7d,
       is_active_30d,
       is_active_90d,
       lifecycle_stage,
       CASE
         WHEN DATEDIFF(snapshot_date, first_seen_date) = 0 THEN 'new'
         WHEN lifecycle_stage = 'returning' THEN 'returning'
         WHEN lifecycle_stage = 'dormant' AND LAG(lifecycle_stage) OVER (PARTITION BY character_id ORDER BY snapshot_date) = 'active' THEN 'dormant'
         WHEN lifecycle_stage = 'churned' AND LAG(lifecycle_stage) OVER (PARTITION BY character_id ORDER BY snapshot_date) = 'dormant' THEN 'churned'
         ELSE NULL
       END AS lifecycle_event,
       in_guild,
       owns_house,
       is_married,
       loyalty_title,
       achievement_points
  FROM stage_calc;

In [0]:
MERGE INTO tibia_analytics.gold.characters_behavior_daily AS target
USING tibia_analytics.gold.tmp_characters_behavior_daily AS source
   ON target.snapshot_date = source.snapshot_date
  AND target.character_id  = source.character_id
WHEN NOT MATCHED THEN 
INSERT (
  snapshot_date,
  character_id,
  world_id,
  account_status,
  sex,
  vocation,
  level,
  level_delta_1d,
  level_delta_7d,
  level_delta_30d,
  level_delta_90d,
  first_seen_date,
  last_login_date,
  days_since_last_login,
  is_active_1d,
  is_active_7d,
  is_active_30d,
  is_active_90d,
  lifecycle_stage,
  lifecycle_event,
  in_guild,
  owns_house,
  is_married,
  loyalty_title,
  achievement_points,
  processed_at
)
VALUES (
  source.snapshot_date,
  source.character_id,
  source.world_id,
  source.account_status,
  source.sex,
  source.vocation,
  source.level,
  source.level_delta_1d,
  source.level_delta_7d,
  source.level_delta_30d,
  source.level_delta_90d,
  source.first_seen_date,
  source.last_login_date,
  source.days_since_last_login,
  source.is_active_1d,
  source.is_active_7d,
  source.is_active_30d,
  source.is_active_90d,
  source.lifecycle_stage,
  source.lifecycle_event,
  source.in_guild,
  source.owns_house,
  source.is_married,
  source.loyalty_title,
  source.achievement_points,
  CURRENT_TIMESTAMP()
);

In [0]:
DROP TABLE IF EXISTS tibia_analytics.gold.tmp_characters_behavior_daily;

In [0]:
CREATE TABLE IF NOT EXISTS tibia_analytics.gold.tmp_characters_behavior_periodic AS
WITH period_base AS (
  SELECT 'Day' AS granularity,
         snapshot_date AS period_start,
         snapshot_date AS period_end,
         1 AS period_days,
         snapshot_date,
         character_id,
         world_id,
         account_status,
         sex,
         vocation,
         level,
         first_seen_date,
         last_login_date,
         days_since_last_login,
         is_active_1d,
         is_active_7d,
         is_active_30d,
         is_active_90d,
         lifecycle_stage,
         lifecycle_event,
         in_guild,
         owns_house,
         is_married,
         loyalty_title,
         achievement_points
    FROM tibia_analytics.gold.characters_behavior_daily
   UNION ALL
  SELECT 'Week' AS granularity,
         CAST(DATE_TRUNC('WEEK', snapshot_date) AS DATE) AS period_start,
         DATE_ADD(CAST(DATE_TRUNC('WEEK', snapshot_date) AS DATE), 6) AS period_end,
         7 AS period_days,
         snapshot_date,
         character_id,
         world_id,
         account_status,
         sex,
         vocation,
         level,
         first_seen_date,
         last_login_date,
         days_since_last_login,
         is_active_1d,
         is_active_7d,
         is_active_30d,
         is_active_90d,
         lifecycle_stage,
         lifecycle_event,
         in_guild,
         owns_house,
         is_married,
         loyalty_title,
         achievement_points
    FROM tibia_analytics.gold.characters_behavior_daily
   UNION ALL
  SELECT 'Month' AS granularity,
         CAST(DATE_TRUNC('MONTH', snapshot_date) AS DATE) AS period_start,
         LAST_DAY(snapshot_date) AS period_end,
         DAY(LAST_DAY(snapshot_date)) AS period_days,
         snapshot_date,
         character_id,
         world_id,
         account_status,
         sex,
         vocation,
         level,
         first_seen_date,
         last_login_date,
         days_since_last_login,
         is_active_1d,
         is_active_7d,
         is_active_30d,
         is_active_90d,
         lifecycle_stage,
         lifecycle_event,
         in_guild,
         owns_house,
         is_married,
         loyalty_title,
         achievement_points
    FROM tibia_analytics.gold.characters_behavior_daily
   UNION ALL
  SELECT 'Quarter' AS granularity,
         CAST(DATE_TRUNC('QUARTER', snapshot_date) AS DATE) AS period_start,
         DATE_SUB(ADD_MONTHS(CAST(DATE_TRUNC('QUARTER', snapshot_date) AS DATE), 3), 1) AS period_end,
         DATEDIFF(ADD_MONTHS(DATE_TRUNC('QUARTER', snapshot_date), 3), DATE_TRUNC('QUARTER', snapshot_date)) AS period_days,
         snapshot_date,
         character_id,
         world_id,
         account_status,
         sex,
         vocation,
         level,
         first_seen_date,
         last_login_date,
         days_since_last_login,
         is_active_1d,
         is_active_7d,
         is_active_30d,
         is_active_90d,
         lifecycle_stage,
         lifecycle_event,
         in_guild,
         owns_house,
         is_married,
         loyalty_title,
         achievement_points
    FROM tibia_analytics.gold.characters_behavior_daily
),
period_metadata AS (
  SELECT granularity,
         period_start,
         COUNT(DISTINCT snapshot_date) AS observed_days,
         MIN(snapshot_date) AS first_snapshot_date,
         MAX(snapshot_date) AS last_snapshot_date
    FROM period_base
   GROUP BY granularity,
            period_start
),
max_snapshot AS (
  SELECT MAX(snapshot_date) AS max_date
    FROM tibia_analytics.gold.characters_behavior_daily
),
period_progression AS (
  SELECT granularity,
         period_start,
         character_id,
         MIN_BY(level, snapshot_date) AS level_start,
         MAX_BY(level, snapshot_date) - MIN_BY(level, snapshot_date) AS level_delta_period
    FROM period_base
   GROUP BY granularity,
            period_start,
            character_id
),
period_activity AS (
  SELECT granularity,
         period_start,
         character_id,
         TRUE AS is_active_in_period
    FROM period_base
   WHERE last_login_date BETWEEN period_start AND period_end
   GROUP BY granularity,
            period_start,
            character_id
),
period_event_priority AS (
  SELECT granularity,
         period_start,
         character_id,
         CASE
           WHEN MAX(CASE WHEN lifecycle_event = 'new'       THEN 1 ELSE 0 END) = 1 THEN 'new'
           WHEN MAX(CASE WHEN lifecycle_event = 'returning' THEN 1 ELSE 0 END) = 1 THEN 'returning'
           WHEN MAX(CASE WHEN lifecycle_event = 'dormant'   THEN 1 ELSE 0 END) = 1 THEN 'dormant'
           WHEN MAX(CASE WHEN lifecycle_event = 'churned'   THEN 1 ELSE 0 END) = 1 THEN 'churned'
           ELSE NULL
         END AS lifecycle_event
    FROM period_base
   GROUP BY granularity,
            period_start,
            character_id
),
period_events AS (
  SELECT granularity,
         period_start,
         character_id,
         SUM(CASE WHEN lifecycle_event = 'new'       THEN 1 ELSE 0 END) AS new_events,
         SUM(CASE WHEN lifecycle_event = 'returning' THEN 1 ELSE 0 END) AS returning_events,
         SUM(CASE WHEN lifecycle_event = 'dormant'   THEN 1 ELSE 0 END) AS dormant_events,
         SUM(CASE WHEN lifecycle_event = 'churned'   THEN 1 ELSE 0 END) AS churned_events
    FROM period_base
   GROUP BY granularity,
            period_start,
            character_id
),
period_snapshot AS (
  SELECT pb.granularity,
         pb.period_start,
         pb.period_end,
         pb.period_days,
         pm.observed_days,
         CASE
           WHEN pm.observed_days       = pb.period_days  THEN FALSE
           ELSE TRUE
         END AS is_partial_period,
         CASE
           WHEN pm.observed_days       = pb.period_days  THEN 'full'
           WHEN pm.first_snapshot_date > pm.period_start THEN 'partial_start'
           WHEN pb.period_end         >= ms.max_date     THEN 'partial_current'
           ELSE 'partial_missing'
         END AS period_status,
         pb.character_id,
         pb.world_id,
         pb.account_status,
         pb.sex,
         pb.vocation,
         pp.level_start,
         pb.level,
         pp.level_delta_period,
         pb.first_seen_date,
         pb.last_login_date,
         pb.days_since_last_login,
         COALESCE(pa.is_active_in_period, FALSE) AS is_active_in_period,
         pb.is_active_1d,
         pb.is_active_7d,
         pb.is_active_30d,
         pb.is_active_90d,
         pb.lifecycle_stage,
         pep.lifecycle_event,
         pe.new_events,
         pe.returning_events,
         pe.dormant_events,
         pe.churned_events,
         pb.in_guild,
         pb.owns_house,
         pb.is_married,
         pb.loyalty_title,
         pb.achievement_points
    FROM period_base AS pb
   CROSS JOIN max_snapshot AS ms
    LEFT JOIN period_metadata AS pm
      ON pm.granularity = pb.granularity
     AND pm.period_start = pb.period_start
    LEFT JOIN period_progression AS pp
      ON pp.granularity = pb.granularity
     AND pp.period_start = pb.period_start
     AND pp.character_id = pb.character_id
    LEFT JOIN period_activity AS pa
      ON pa.granularity = pb.granularity
     AND pa.period_start = pb.period_start
     AND pa.character_id = pb.character_id
    LEFT JOIN period_event_priority AS pep
      ON pep.granularity = pb.granularity
     AND pep.period_start = pb.period_start
     AND pep.character_id = pb.character_id
    LEFT JOIN period_events AS pe
      ON pe.granularity = pb.granularity
     AND pe.period_start = pb.period_start
     AND pe.character_id = pb.character_id
 QUALIFY ROW_NUMBER() OVER (
           PARTITION BY pb.granularity, pb.character_id, pb.period_start
               ORDER BY pb.snapshot_date DESC
         ) = 1
),
final_dataset AS (
  SELECT ps.granularity,
         ps.period_start,
         ps.period_end,
         ps.period_days,
         ps.observed_days,
         ps.is_partial_period,
         ps.period_status,
         ps.character_id,
         ps.world_id,
         ps.account_status,
         ps.sex,
         ps.vocation,
         ps.level_start,
         ps.level,
         ps.level_delta_period,
         ps.level - prev_short.level  AS level_delta_short,
         ps.level - prev_medium.level AS level_delta_medium,
         ps.level - prev_long.level   AS level_delta_long,
         ps.level - prev_xlong.level  AS level_delta_xlong,
         ps.first_seen_date,
         ps.last_login_date,
         ps.days_since_last_login,
         ps.is_active_in_period,
         ps.is_active_1d,
         ps.is_active_7d,
         ps.is_active_30d,
         ps.is_active_90d,
         ps.lifecycle_stage,
         ps.lifecycle_event,
         ps.new_events,
         ps.returning_events,
         ps.dormant_events,
         ps.churned_events,
         ps.in_guild,
         ps.owns_house,
         ps.is_married,
         ps.loyalty_title,
         ps.achievement_points
    FROM period_snapshot AS ps
     LEFT JOIN period_snapshot AS prev_short
       ON prev_short.granularity = ps.granularity
      AND prev_short.character_id = ps.character_id
      AND prev_short.period_start = CASE
                                      WHEN ps.granularity = 'Day'     THEN DATE_SUB(ps.period_start, 1)
                                      WHEN ps.granularity = 'Week'    THEN DATE_SUB(ps.period_start, 7)
                                      WHEN ps.granularity = 'Month'   THEN ADD_MONTHS(ps.period_start, -1)
                                      WHEN ps.granularity = 'Quarter' THEN ADD_MONTHS(ps.period_start, -3)
                                    END
     LEFT JOIN period_snapshot AS prev_medium
       ON prev_medium.granularity = ps.granularity
      AND prev_medium.character_id = ps.character_id
      AND prev_medium.period_start = CASE
                                      WHEN ps.granularity = 'Day'     THEN DATE_SUB(ps.period_start, 7)
                                      WHEN ps.granularity = 'Week'    THEN DATE_SUB(ps.period_start, 28)
                                      WHEN ps.granularity = 'Month'   THEN ADD_MONTHS(ps.period_start, -3)
                                      WHEN ps.granularity = 'Quarter' THEN ADD_MONTHS(ps.period_start, -12)
                                    END
     LEFT JOIN period_snapshot AS prev_long
       ON prev_long.granularity = ps.granularity
      AND prev_long.character_id = ps.character_id
      AND prev_long.period_start = CASE
                                      WHEN ps.granularity = 'Day'     THEN DATE_SUB(ps.period_start, 30)
                                      WHEN ps.granularity = 'Week'    THEN DATE_SUB(ps.period_start, 84)
                                      WHEN ps.granularity = 'Month'   THEN ADD_MONTHS(ps.period_start, -12)
                                      ELSE NULL
                                    END
     LEFT JOIN period_snapshot AS prev_xlong
       ON prev_xlong.granularity = ps.granularity
      AND prev_xlong.character_id = ps.character_id
      AND prev_xlong.period_start = CASE
                                      WHEN ps.granularity = 'Day'     THEN DATE_SUB(ps.period_start, 90)
                                      WHEN ps.granularity = 'Week'    THEN DATE_SUB(ps.period_start, 364)
                                      ELSE NULL
                                    END
)
SELECT *
  FROM final_dataset

In [0]:
TRUNCATE TABLE tibia_analytics.gold.characters_behavior_periodic;

In [0]:
INSERT INTO tibia_analytics.gold.characters_behavior_periodic (
  granularity,
  period_start,
  period_end,
  period_days,
  observed_days,
  is_partial_period,
  period_status,
  character_id,
  world_id,
  account_status,
  sex,
  vocation,
  level_start,
  level,
  level_delta_period,
  level_delta_short,
  level_delta_medium,
  level_delta_long,
  level_delta_xlong,
  first_seen_date,
  last_login_date,
  days_since_last_login,
  is_active_in_period,
  is_active_1d,
  is_active_7d,
  is_active_30d,
  is_active_90d,
  lifecycle_stage,
  lifecycle_event,
  new_events,
  returning_events,
  dormant_events,
  churned_events,
  in_guild,
  owns_house,
  is_married,
  loyalty_title,
  achievement_points,
  processed_at
)
SELECT granularity,
       period_start,
       period_end,
       period_days,
       observed_days,
       is_partial_period,
       period_status,
       character_id,
       world_id,
       account_status,
       sex,
       vocation,
       level_start,
       level,
       level_delta_period,
       level_delta_short,
       level_delta_medium,
       level_delta_long,
       level_delta_xlong,
       first_seen_date,
       last_login_date,
       days_since_last_login,
       is_active_in_period,
       is_active_1d,
       is_active_7d,
       is_active_30d,
       is_active_90d,
       lifecycle_stage,
       lifecycle_event,
       new_events,
       returning_events,
       dormant_events,
       churned_events,
       in_guild,
       owns_house,
       is_married,
       loyalty_title,
       achievement_points,
       CURRENT_TIMESTAMP()
  FROM tibia_analytics.gold.tmp_characters_behavior_periodic;

In [0]:
DROP TABLE IF EXISTS tibia_analytics.gold.tmp_characters_behavior_periodic;